In [ ]:
import os, sys, subprocess, tarfile, shutil

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print("FAILED:", cmd, r.stdout[-500:], r.stderr[-500:])
    return r

print("python:", sys.version.split()[0])
sh(f"{sys.executable} -m pip install -q python-chess zstandard huggingface_hub onnxruntime pytest")
import chess
print("python-chess:", chess.__version__)

WORK = "/kaggle/working"
IN = None
for cand in ["/kaggle/input/datasets/omarcodeplay/magnusv1-src",
             "/kaggle/input/magnusv1-src"]:
    if os.path.isdir(cand):
        IN = cand
        break
assert IN, "source dataset not mounted"
print("dataset root:", IN)

if os.path.isdir(os.path.join(IN, "kaggle", "src")) and \
   not os.path.isdir(os.path.join(WORK, "kaggle", "src")):
    shutil.copytree(IN, WORK, dirs_exist_ok=True)
    print("copied src")
print("src:", sorted(os.listdir(os.path.join(WORK, "kaggle", "src"))))

In [ ]:
import os
if os.getenv("KAGGLE_URL_BASE") == "https://api.kaggle.com":
    os.environ["KAGGLE_URL_BASE"] = "https://www.kaggle.com"
print("KAGGLE_URL_BASE:", os.getenv("KAGGLE_URL_BASE"))

try:
    from kaggle_secrets import UserSecretsClient
    us = UserSecretsClient()
    for name in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN"]:
        try:
            v = us.get_secret(name)
            os.environ["HF_TOKEN"] = v
            os.environ.setdefault("HF_USER", "OmarCodePlay")
            print(f"  HF_TOKEN FOUND via {name!r} (len {len(v)})")
            break
        except Exception as e:
            print(f"  {name}: {type(e).__name__}")
except Exception as e:
    print("kaggle_secrets unavailable:", e)

In [ ]:
import os, sys
sys.path.insert(0, "/kaggle/working/kaggle/src")
os.makedirs("/kaggle/working/ckpt", exist_ok=True)
import sync
print("HF_USER:", sync.USER, "| repo:", sync.MODEL_REPO)

for name in ["best_v9.pt", "engine.pt", "best_v11.pt"]:
    try:
        p = sync.pull_ckpt(name, "/kaggle/working/ckpt")
        if p and os.path.exists(p):
            print(f"  {name:14s} -> {p}  ({os.path.getsize(p)/1e6:.1f}MB)")
        else:
            print(f"  {name:14s} -> not on HF")
    except Exception as e:
        print(f"  {name:14s} -> {type(e).__name__}: {str(e)[:120]}")

In [ ]:
import os, sys, subprocess, hashlib
EXPECT = {
    "encode.py": "d4ec60ca5594a5a2b094329804453cde04433133475cb5cc54e3313a490de9ab",
    "model.py":  "cd5450747ae37564d20b4d88430dbfa28b47ac9752a374b52820db838f287ada",
}
ok = True
for fn, want in EXPECT.items():
    p = os.path.join("/kaggle/working/kaggle/src", fn)
    got = hashlib.sha256(open(p, "rb").read()).hexdigest()
    m = (got == want); ok &= m
    print(f"  {fn:10s} {'OK' if m else 'MISMATCH'}  {got[:16]}...")
assert ok, "FREEZE ANCHOR FAILED — do not proceed"

r = subprocess.run([sys.executable, "/kaggle/working/tests/test_encode.py"],
                   capture_output=True, text=True)
print(r.stdout[-800:])
if r.returncode or "PASS" not in r.stdout:
    print(r.stderr[-800:])
    raise SystemExit("M0 TESTS FAILED")
print("freeze + M0: PASS")

In [ ]:
# DIAGNOSTIC: v9 vs v11 top1 → sets RESTART_FROM_V9 for cell 7
import sys, os, torch, numpy as np
sys.path.insert(0, "/kaggle/working/kaggle/src")

VAL = None
for cand in ["/kaggle/input/magnusv1-data/val.npz",
             "/kaggle/input/datasets/omarcodeplay/magnusv1-data/val.npz"]:
    if os.path.exists(cand): VAL = cand; break
print("val:", VAL)

RESTART_FROM_V9 = True

if VAL:
    val = np.load(VAL)
    N = min(5000, len(val["bb"]))
    bb   = torch.from_numpy(val["bb"][:N].copy())
    meta = torch.from_numpy(val["meta"][:N].copy())
    mvk0 = torch.from_numpy(val["mvk"][:N, 0].copy()).long()

    wk = val["wk"]
    pure = (wk[:, 1] == 0).mean()
    print(f"shard: pure(wk1=0)={pure:.3f}  (target 0.05–0.10)")

    def load_net(path):
        ck = torch.load(path, map_location="cpu", weights_only=False)
        is_v11 = any("v2_wdl" in k for k in ck["sd"])
        if is_v11:
            from model_v11 import ChessNetV11 as NetCls
        else:
            from model import ChessNet as NetCls
        net = NetCls(ck["filters"], ck["blocks"])
        net.load_state_dict({k.replace("module.",""): v for k,v in ck["sd"].items()})
        return net.cuda().half().eval(), ck

    def top1(net, B=512):
        correct = 0
        with torch.inference_mode():
            for i in range(0, len(bb), B):
                out = net(bb[i:i+B].cuda(), meta[i:i+B].cuda())
                pol = out[0] if isinstance(out, tuple) else out
                correct += (pol.argmax(dim=1).cpu() == mvk0[i:i+B]).sum().item()
        return correct / len(bb)

    results = {}
    for tag, path in [("v9", "/kaggle/working/ckpt/best_v9.pt"),
                      ("v11", "/kaggle/working/ckpt/best_v11.pt")]:
        if not os.path.exists(path): print(f"{tag}: MISSING"); continue
        net, ck = load_net(path)
        acc = top1(net)
        results[tag] = acc
        print(f"{tag:4s}  step={ck.get('step')}  top1={acc:.4f}")
        del net; torch.cuda.empty_cache()

    if "v9" in results and "v11" in results:
        d = results["v11"] - results["v9"]
        RESTART_FROM_V9 = (d < -0.003)
        print(f"\ndelta(v11-v9)={d:+.4f}  gate(>=-0.003): {not RESTART_FROM_V9}")
        print(">>>", "RESTART from v9" if RESTART_FROM_V9 else "RESUME v11")
    else:
        print("\n>>> defaulting to RESTART from v9 (v11 missing)")

print("\nRESTART_FROM_V9 =", RESTART_FROM_V9)

In [ ]:
# DIAGNOSTIC: does train_v11.py accept --mirror? → sets MIRROR_ARG for cell 7
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/scripts/train_v11.py", "--help"],
                   capture_output=True, text=True)
HELP = r.stdout + r.stderr
print(HELP[:2500])
MIRROR_ARG = "--mirror" in HELP
print(f"\n>>> --mirror available: {MIRROR_ARG}")

In [ ]:
# Fix M3b openings path — write to BOTH locations fastchess might resolve to
import os
epds = [
    "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1",
    "rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "rnbqkb1r/pppppppp/5n2/8/8/5N2/PPPPPPPP/RNBQKB1R w KQkq - 2 2",
    "rnbqkbnr/pp1ppppp/8/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2",
    "rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2",
    "rnbqkb1r/pp1ppppp/8/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 1 3",
]
for p in ["/kaggle/working/data/openings/1.csv",
          "/kaggle/working/kaggle/data/openings/1.csv"]:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, "w").write("\n".join(epds) + "\n")
    print(f"wrote {p}  exists={os.path.exists(p)}")

In [ ]:
# TRAINING — streams log lines. Reads RESTART_FROM_V9 (cell 4) and MIRROR_ARG (cell 5).
import subprocess, sys, os, torch

SHARDS = None
for cand in ["/kaggle/input/magnusv1-data",
             "/kaggle/input/datasets/omarcodeplay/magnusv1-data"]:
    if os.path.isdir(cand): SHARDS = cand; break
assert SHARDS, "shards not mounted"
print("shards:", SHARDS)

CK = "/kaggle/working/ckpt/best_v11.pt"

init_args   = ["--init", "/kaggle/working/ckpt/best_v9.pt"]
resume_args = []
if RESTART_FROM_V9:
    if os.path.exists(CK):
        os.remove(CK)
        print(">>> deleted best_v11.pt (fresh restart)")
    print(">>> mode: RESTART from v9")
elif os.path.exists(CK):
    resume_args = ["--resume", CK]
    print(f">>> mode: RESUME best_v11.pt (step={torch.load(CK, map_location='cpu', weights_only=False).get('step')})")

mirror_args = ["--mirror", "0.5"] if MIRROR_ARG else []
if not MIRROR_ARG:
    print(">>> WARNING: --mirror not supported — running WITHOUT augmentation")

cmd = [sys.executable, "/kaggle/working/scripts/train_v11.py",
       "--src-dir", "/kaggle/working/kaggle/src",
       "--shards", SHARDS,
       "--out", CK,
       "--steps", "150000",
       "--push-hf-every", "5000",
       "--log", "/kaggle/working/train_v11.log",
       ] + init_args + resume_args + mirror_args
print("cmd:", " ".join(cmd))
print("=" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
print(f"\n=== training exited rc={rc} ===")

try:
    import sync
    if os.path.exists(CK):
        sync.push_ckpt(CK, "best_v11.pt")
        print("pushed final best_v11.pt")
except Exception as e:
    print("final push failed:", e)